In [8]:
#Import packages and load data

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")

url = "https://download.medicaid.gov/data/sdud-2025-updated-dec2025.csv"
df = pd.read_csv(url)

df.head()

,Utilization Type,State,NDC,Labeler Code,Product Code,Package Size,Year,Quarter,Suppression Used,Product Name,Units Reimbursed,Number of Prescriptions,Total Amount Reimbursed,Medicaid Amount Reimbursed,Non Medicaid Amount Reimbursed
0,FFSU,AK,2143380,2,1433,80,2025,2,False,TRULICITY,216.0,107.0,102976.40,98630.87,4345.53
1,FFSU,AK,2143480,2,1434,80,2025,2,False,TRULICITY,218.0,109.0,104481.92,101806.64,2675.28
2,FFSU,AK,2143611,2,1436,11,2025,2,False,EMGALITY P,21.0,20.0,15227.25,15227.25,0.00
3,FFSU,AK,2144511,2,1445,11,2025,2,False,TALTZ AUTO,33.0,30.0,231532.28,231532.28,0.00
4,FFSU,AK,2144527,2,1445,27,2025,2,True,TALTZ AUTO,NaN,NaN,NaN,NaN,NaN


In [9]:
# Checking column names
df.columns

Index(['Utilization Type', 'State', 'NDC', 'Labeler Code', 'Product Code',
       'Package Size', 'Year', 'Quarter', 'Suppression Used', 'Product Name',
       'Units Reimbursed', 'Number of Prescriptions',
       'Total Amount Reimbursed', 'Medicaid Amount Reimbursed',
       'Non Medicaid Amount Reimbursed'],
      dtype='object')

In [10]:
#Clean data by taking non-zero values
df = df.dropna(subset=["Total Amount Reimbursed", "Product Name"])
df = df[df["Total Amount Reimbursed"] > 0]

#Aggregate the top drugs by spending
top_spending_drugs = (
    df.groupby("Product Name")["Total Amount Reimbursed"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

In [11]:
import plotly.graph_objects as go

def bucket_spending(x):
    if x < 500_000_000:
        return "0–500M"
    elif x < 1_000_000_000:
        return "500M–1B"
    elif x < 1_500_000_000:
        return "1B–1.5B"
    else:
        return "1.5B+"

top_spending_drugs["Spending Bucket"] = top_spending_drugs["Total Amount Reimbursed"].apply(bucket_spending)

color_map = {
    "0–500M": "lightblue",
    "500M–1B": "orange",
    "1B–1.5B": "red",
    "1.5B+": "darkred"
}

fig = go.Figure()

# Create one bar trace per bucket so the legend works
for bucket in ["1.5B+", "1B–1.5B", "500M–1B", "0–500M"]:
    bucket_df = top_spending_drugs[top_spending_drugs["Spending Bucket"] == bucket]

    fig.add_trace(go.Bar(
        x=bucket_df["Total Amount Reimbursed"],
        y=bucket_df["Product Name"],
        orientation="h",
        name=bucket,
        marker=dict(color=color_map[bucket]),
        hovertemplate=
            "Drug: %{y}<br>" +
            "Cost: $%{x:,.0f}<br>" +
            "Bucket: " + bucket +
            "<extra></extra>"
    ))

fig.update_layout(
    title="Top Drugs by Spending",
    xaxis=dict(title="Cost ($)"),
    yaxis=dict(
        title="Drug Name",
        categoryorder="total ascending"
    ),
    height=700,
    showlegend=True,
    legend_title_text="Spending Bucket"
)

fig.show()